# Phase 1 — Options Fundamentals and Black-Scholes

**Project:** Neural Network-Based Options Pricing

This notebook is the starting point. We will work with one European option, understand each input, implement Black-Scholes, calculate Greeks, and check put-call parity.

### Learning objectives
1. Explain basic option-contract terminology.
2. Calculate European call and put prices.
3. Calculate the main option Greeks.
4. Check put-call parity.
5. Explore how prices respond to changes in model inputs.

**Conventions:** Annualized volatility and rates; time to maturity in years; continuously compounded risk-free rate. Our first example uses 30/365 years.


## 1. Option-contract fundamentals

A **call** gives its holder the right, but not the obligation, to buy the underlying asset at the strike price. A **put** gives the right, but not the obligation, to sell it.

- **Underlying price (S):** current asset price.
- **Strike (K):** contract price at which the asset can be bought or sold.
- **Expiration (T):** time remaining until the contract expires.
- **Volatility (σ):** annualized uncertainty in the underlying asset's returns.
- **Risk-free rate (r):** annualized rate used for discounting.
- **Premium:** price paid to purchase the option.

A European option can be exercised only at expiration. Black-Scholes here is for European options.


## 2. Set up the first example

Illustrative assumptions (not live market quotes):
- Stock price: $100
- Strike price: $100
- Time to expiration: 30 calendar days
- Annualized volatility: 20%
- Annualized risk-free rate: 5%
- Dividend yield: 0%


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

S = 100.0
K = 100.0
days = 30
T = days / 365
r = 0.05
sigma = 0.20
q = 0.00

print(f"Time to expiration: {T:.6f} years")


## 3. Payoff at expiration

At expiration, a call buyer's payoff is `max(S_T - K, 0)`. A put buyer's payoff is `max(K - S_T, 0)`.

Payoff is not the same as profit: profit also subtracts the premium paid.


In [ ]:
terminal_prices = np.linspace(60, 140, 161)
call_payoff = np.maximum(terminal_prices - K, 0)
put_payoff = np.maximum(K - terminal_prices, 0)

plt.figure(figsize=(9, 5))
plt.plot(terminal_prices, call_payoff, label="Call payoff")
plt.plot(terminal_prices, put_payoff, label="Put payoff")
plt.axvline(K, linestyle="--", label="Strike")
plt.xlabel("Underlying price at expiration")
plt.ylabel("Payoff per share")
plt.title("European option payoffs at expiration")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 4. Implement Black-Scholes

For European options with continuous dividend yield `q`:

`d1 = [ln(S/K) + (r - q + σ²/2)T] / (σ√T)`

`d2 = d1 - σ√T`

`Call = S exp(-qT) N(d1) - K exp(-rT) N(d2)`

`Put = K exp(-rT) N(-d2) - S exp(-qT) N(-d1)`

`N(x)` is the standard normal cumulative distribution function.


In [ ]:
def black_scholes(S, K, T, r, sigma, q=0.0):
    """European Black-Scholes call and put prices.

    Rates and volatility are annualized; T is measured in years.
    """
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        raise ValueError("S, K, T, and sigma must be positive.")

    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    call = S * np.exp(-q*T) * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)
    put = K * np.exp(-r*T) * norm.cdf(-d2) - S * np.exp(-q*T) * norm.cdf(-d1)

    return {"call": call, "put": put, "d1": d1, "d2": d2}

prices = black_scholes(S, K, T, r, sigma, q)
prices


### Interpret the result

Write down:
- What are the call and put prices?
- Why are they not necessarily equal when the strike equals the stock price?
- Which inputs do you expect to have the largest effect?

Do not move on until you can explain each input.


## 5. Calculate the Greeks

- **Delta:** sensitivity to the underlying price.
- **Gamma:** change in delta as the underlying price changes.
- **Vega:** sensitivity to volatility.
- **Theta:** sensitivity to the passage of time.
- **Rho:** sensitivity to the risk-free rate.

Vega and rho below are sensitivities to a 1.00 change in decimal volatility/rate. Divide by 100 for the approximate sensitivity to a one-percentage-point change. Theta is shown per calendar day.


In [ ]:
def black_scholes_greeks(S, K, T, r, sigma, q=0.0):
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        raise ValueError("S, K, T, and sigma must be positive.")

    d1 = (np.log(S/K) + (r-q+0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    pdf = norm.pdf(d1)

    call_delta = np.exp(-q*T) * norm.cdf(d1)
    put_delta = np.exp(-q*T) * (norm.cdf(d1)-1)
    gamma = np.exp(-q*T) * pdf / (S*sigma*np.sqrt(T))
    vega = S*np.exp(-q*T) * pdf*np.sqrt(T)

    call_theta = (
        -S*np.exp(-q*T)*pdf*sigma/(2*np.sqrt(T))
        - r*K*np.exp(-r*T)*norm.cdf(d2)
        + q*S*np.exp(-q*T)*norm.cdf(d1)
    )
    put_theta = (
        -S*np.exp(-q*T)*pdf*sigma/(2*np.sqrt(T))
        + r*K*np.exp(-r*T)*norm.cdf(-d2)
        - q*S*np.exp(-q*T)*norm.cdf(-d1)
    )

    call_rho = K*T*np.exp(-r*T)*norm.cdf(d2)
    put_rho = -K*T*np.exp(-r*T)*norm.cdf(-d2)

    return pd.DataFrame({
        "Call": {
            "Delta": call_delta,
            "Gamma": gamma,
            "Vega (1.00 vol)": vega,
            "Theta (per day)": call_theta/365,
            "Rho (1.00 rate)": call_rho
        },
        "Put": {
            "Delta": put_delta,
            "Gamma": gamma,
            "Vega (1.00 vol)": vega,
            "Theta (per day)": put_theta/365,
            "Rho (1.00 rate)": put_rho
        }
    })

greeks = black_scholes_greeks(S, K, T, r, sigma, q)
greeks


## 6. Verify put-call parity

For European options with continuous dividend yield:

`Call - Put = S exp(-qT) - K exp(-rT)`

This is a useful implementation check.


In [ ]:
parity_left = prices["call"] - prices["put"]
parity_right = S*np.exp(-q*T) - K*np.exp(-r*T)

print("Call - Put:", parity_left)
print("S*exp(-qT) - K*exp(-rT):", parity_right)
print("Absolute difference:", abs(parity_left-parity_right))

assert np.isclose(parity_left, parity_right, atol=1e-10)


## 7. Sensitivity to volatility

Before running the cell, predict what happens to a call price as volatility increases. Then inspect the plot.


In [ ]:
volatilities = np.linspace(0.05, 0.80, 100)
call_prices = [black_scholes(S, K, T, r, vol, q)["call"] for vol in volatilities]

plt.figure(figsize=(9, 5))
plt.plot(volatilities, call_prices)
plt.xlabel("Annualized volatility")
plt.ylabel("European call price")
plt.title("Call price sensitivity to volatility")
plt.grid(alpha=0.3)
plt.show()


## 8. Exercises

1. Change the stock price from $100 to $110. What happens to call and put prices?
2. Change expiration from 30 days to 180 days. What happens? Is the effect always identical for calls and puts?
3. Increase volatility from 20% to 40%. Why do both calls and puts generally become more valuable?
4. Set the risk-free rate to 0%. How do prices change?
5. Change the strike to $110. Is the call in-, at-, or out-of-the-money?
6. Explain the difference between payoff and profit.
7. Explain why put-call parity should hold for European options.

**Optional coding challenge:** Create a function that accepts a list of strike prices and returns call prices, put prices, and Greeks for each strike.


## 9. Phase 1 completion checklist

- [ ] I can explain calls, puts, strike, expiration, and premium.
- [ ] I understand Black-Scholes inputs and assumptions.
- [ ] I can implement call and put prices in Python.
- [ ] I can explain Delta, Gamma, Theta, Vega, and Rho.
- [ ] I verified put-call parity.
- [ ] I completed the sensitivity exercises.
- [ ] I can explain limitations of Black-Scholes.

**Next phase:** Monte Carlo simulation and geometric Brownian motion. We will simulate possible stock-price paths and compare the estimated option price with Black-Scholes.
